In [1]:
import numpy as np
from scipy.stats import multivariate_normal
from tqdm import tqdm
import time

In [2]:
def empirical_cdf(x, y, X, Y):
    return np.mean((X <= x) & (Y <= y))

def G_uniform(x, y):
    return x * y

def ks_2d_statistic(X, Y, G):

    n = len(X)
    X = np.clip(X,0,1)
    Y = np.clip(Y,0,1)
    
    # maximum of observed points
    D1 = max(
        empirical_cdf(X[i], Y[i], X, Y) - G(X[i], Y[i])
        for i in range(n)
    )

    # maximum distance over all intersection points
    D2 = max(
            empirical_cdf(X[j], Y[i], X, Y) - G(X[j], Y[i])
            for j in range(n) for i in range(n) 
            if (X[j]>X[i] and Y[j]<Y[i])
        )

    # minimum distance over all intersection poitns (with 2/n correction)
    D3 = (2 / n) - min(
                        empirical_cdf(X[j], Y[i], X, Y) - G(X[j], Y[i])
                        for i in range(n) for j in range(n)
                        if (X[j] > X[i] and Y[j] < Y[i])
    )

    # maximum distance among projections of observed points on the right boundary (x = 1)
    D4 = (1/n) - min(
                    empirical_cdf(1, Y[i], X, Y) - G(1, Y[i])
                    for i in range(n)
    )

    # maximum distance among projections of the observed points on top boundary (y = 1)
    D5 = (1/n) - min(
                    empirical_cdf(X[i], 1, X, Y) - G(X[i], 1)
                    for i in range(n)
    )

    # final statistic
    Dn = max(D1, D2, D3, D4, D5)

    return Dn

In [3]:
mean0 = np.array([0,0])
cov = np.array([[1, 0.5], [0.5, 1]])

In [17]:
# alpha quantiles
n = 100
np.random.seed(0)
Dn_null = []
for _ in tqdm(range(2000)):
    sample = np.random.uniform(0, 1, size=(n, 2)) #sample from two dim. uniform 0-1
    X0, Y0 = sample[:,0], sample[:,1] # split x and y in respective vector
    Dn_null.append(ks_2d_statistic(X0, Y0, G_uniform))
c_alpha = np.quantile(Dn_null, 0.95)
print(c_alpha)

100%|██████████| 2000/2000 [01:19<00:00, 25.19it/s]

0.16631481150951294
